# Orchid disease and severity — improvement experiments

**R26-SE-018 · disease detection, severity and treatment**

This notebook answers the PP2 panel: *"you didn't try to improve the accuracy or check other metrics."*
It trains the experiments one after another. Each one changes **one** thing in the recipe and is scored
on the **validation** photos, so you can show exactly what each choice was worth. The **test** photos
are not used here at all.

| Step | What it does | Time |
|---|---|---|
| 1 | Confirm a GPU is attached | 10 s |
| 2 | Mount Google Drive | 30 s |
| 3 | Unpack `orchid_experiments_colab.zip` | 1 min |
| 4 | Rebuild both splits and the 54× copies, then check for leakage | 8–12 min |
| 5 | Smoke test: proves the code runs on this machine | 3–5 min |
| 6 | **Run all experiments** | **about 2–3 hours** |
| 7 | Build the results table and charts | 1–2 min |
| 8 | Download the results | 1–3 min |

---

## Before you start

**1. Turn on the GPU.** Runtime → Change runtime type → Hardware accelerator → **T4 GPU** → Save.

**2. Upload the zip.** Drag `orchid_experiments_colab.zip` into the **top level of My Drive** (not into a
folder) at [drive.google.com](https://drive.google.com), and wait until the upload finishes.

**3. Keep this tab open and visible** while it runs. Colab disconnects idle sessions.

**If Colab disconnects:** nothing finished is lost. Results are saved to Drive after every experiment.
Reconnect, run Steps 1–4 again, skip Step 5, and run Step 6. It skips every experiment already done.

## Step 1 — confirm the GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU       :', [g.name for g in gpus] or 'NONE')
if not gpus:
    print('\n*** NO GPU *** Runtime -> Change runtime type -> T4 GPU -> Save, then re-run this cell.')
else:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Step 2 — mount Google Drive

A popup asks for permission. Choose your account and allow access.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ZIP = Path('/content/drive/MyDrive/orchid_experiments_colab.zip')
OUT = Path('/content/drive/MyDrive/orchid_experiments')     # results land here
OUT.mkdir(parents=True, exist_ok=True)

if ZIP.exists():
    print('found:', ZIP, '({:.0f} MB)'.format(ZIP.stat().st_size / 1e6))
else:
    print('*** NOT FOUND:', ZIP)
    print('Upload orchid_experiments_colab.zip to the TOP LEVEL of My Drive, wait for it to finish,')
    print('then re-run this cell.')
done = sorted(p.parent.name for p in OUT.glob('*/result.json'))
print('experiments already finished in Drive:', done or 'none yet')

## Step 3 — unpack

Everything goes to `/content`, not to Drive: reading thousands of small files from Drive would starve the GPU.

In [ ]:
import shutil, zipfile, time
from pathlib import Path

t0 = time.time()
for p in ('/content/pkg', '/content/data', '/content/scripts', '/content/models'):
    shutil.rmtree(p, ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/pkg')

Path('/content/data').mkdir()
shutil.move('/content/pkg/processed', '/content/data/processed')
shutil.move('/content/pkg/scripts', '/content/scripts')
shutil.move('/content/pkg/models', '/content/models')
shutil.move('/content/pkg/data_extra/severity_labels.csv', '/content/data/severity_labels.csv')

total = sum(len(list(d.glob('*.jpg'))) for d in Path('/content/data/processed').iterdir())
print('unpacked in {:.0f}s: {} photographs'.format(time.time() - t0, total))
assert total == 667, 'Expected 667 photographs, found {} -- re-upload the zip'.format(total)
print('deployed models:', sorted(p.name for p in Path('/content/models').iterdir()))
print('OK')

## Step 4 — rebuild the splits and the 54× copies

* `split_dataset.py` with seed 42 gives **exactly the same split** as the deployed model used (533 / 67 / 67).
* `split_severity.py` builds the **severity-balanced split** (341 / 43 / 43) used only by the severity experiments.
* Only **training** photos are copied 54 times. Validation and test stay unmodified originals.
* Five copying jobs run side by side to use both of Colab's processors.
* The leakage checks must both say **PASS**. Screenshot them for the thesis.

In [ ]:
%%time
import subprocess, shutil
from pathlib import Path

!cd /content/scripts && python split_dataset.py | tail -8
!cd /content/scripts && python split_severity.py | tail -32

jobs = []
for cls in ['black_leaf_spot', 'phyllosticta_leaf_spot', 'healthy']:
    jobs.append(('split', 'split_augmented', cls))
for cls in ['black_leaf_spot', 'phyllosticta_leaf_spot']:
    jobs.append(('split_severity', 'split_severity_augmented', cls))

procs = []
for src, dst, cls in jobs:
    log = open('/content/augment_{}_{}.log'.format(dst, cls), 'w')
    procs.append(subprocess.Popen([
        'python', 'augment_dataset.py', 'augment',
        '--input',  '/content/data/{}/train/{}'.format(src, cls),
        '--output', '/content/data/{}/train/{}'.format(dst, cls),
        '--disease', cls, '--labels', '/content/data/severity_labels.csv',
    ], cwd='/content/scripts', stdout=log, stderr=subprocess.STDOUT))
for p in procs:
    p.wait()
assert all(p.returncode == 0 for p in procs), 'an augmentation job failed -- see /content/augment_*.log'

for src, dst in [('split', 'split_augmented'), ('split_severity', 'split_severity_augmented')]:
    for part in ['validation', 'test']:
        target = Path('/content/data') / dst / part
        shutil.rmtree(target, ignore_errors=True)
        shutil.copytree(Path('/content/data') / src / part, target)

for dst in ['split_augmented', 'split_severity_augmented']:
    print('\n===', dst, '===')
    for part in ['train', 'validation', 'test']:
        for d in sorted((Path('/content/data') / dst / part).iterdir()):
            if d.is_dir():
                print('  {:<11} {:<24} {:>6}'.format(part, d.name, len(list(d.glob('*.jpg')))))

In [ ]:
# Leakage checks: both must end with RESULT: PASS
!cd /content/scripts && python check_leakage.py | tail -6
!cd /content/scripts && python check_leakage.py --augmented /content/data/split_severity_augmented --split-manifest /content/data/split_severity/split_manifest.csv | tail -6

## Step 5 — smoke test (3–5 minutes)

One epoch on 48 photos for every experiment. It proves every part of the code works on this machine
before you commit hours of GPU time. **Its numbers mean nothing** and it writes to a separate folder,
not to Drive. Skip this step after a reconnect.

In [ ]:
!cd /content/scripts && PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=2 python run_experiments.py --smoke-test --data-root /content/data --out /content/smoke_out

## Step 6 — run all experiments (about 2–3 hours)

The order puts the most important first, so if time runs out the key answers are already saved:

| Order | ID | Question |
|---|---|---|
| 1 | DEP, SDEP | The deployed models' validation scores (seconds) |
| 2 | R | The deployed recipe, rerun with every epoch scored. Also gives D1 (2 epochs), D2 (5 and 10 epochs), D6 (0 vs 30 layers) and D7 (best-F1 epoch) |
| 3 | D9 | Averaging 8 flipped/rotated views of each photo |
| 4 | SO | Severity, original split: frozen, then fine-tuned |
| 5 | SB | Severity, balanced split: frozen, then fine-tuned |
| 6 | D5 | No class weights |
| 7 | D4 | No 54× copies |
| 8 | D6 | Fine-tune 60 layers instead of 30 |
| 9 | D8 | EfficientNetB0 instead of MobileNetV2 |
| 10 | D3a, D3b | Learning rate 0.0001 and 0.01 |

**The first epoch of R is slow** (about 5 minutes) because it decodes all 28,782 training photos once into a
cache. Every later epoch, and every later experiment, reads the cache and is much faster.

Every line `epoch N stage S val loss … acc … macro F1 …` is a validation score. `<- best loss` marks the
epoch the recipe would keep; `early stop` means training stopped by itself because the score stopped improving.

In [ ]:
!cd /content/scripts && PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=2 python run_experiments.py --data-root /content/data --out /content/drive/MyDrive/orchid_experiments

## Step 7 — results table and charts

Builds the table, four charts and a plain-language summary from what Step 6 saved. It needs no GPU and can
be re-run at any time, even while some experiments are still missing.

In [ ]:
!cd /content/scripts && python run_experiments.py --report --out /content/drive/MyDrive/orchid_experiments

from IPython.display import Image, Markdown, display
from pathlib import Path
rep = Path('/content/drive/MyDrive/orchid_experiments/report')
display(Markdown(rep.joinpath('experiments_table.md').read_text()))
for fig in sorted(rep.glob('fig_*.png')):
    display(Image(str(fig), width=800))

## Step 8 — download

Downloads the whole results folder as one zip (the saved model weights make it a few hundred MB).
If the download stalls, open [drive.google.com](https://drive.google.com), right-click the
`orchid_experiments` folder and choose **Download** instead.

**Do not run a test-set evaluation yourself.** The winner is chosen from the validation scores first, and
then tested exactly once.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/orchid_experiments_results', 'zip',
                    '/content/drive/MyDrive/orchid_experiments')
files.download('/content/orchid_experiments_results.zip')

---

## If something goes wrong

| Symptom | Fix |
|---|---|
| `NOT FOUND: orchid_experiments_colab.zip` | The zip must be at the top level of My Drive, fully uploaded. |
| No GPU in Step 1 | Runtime → Change runtime type → T4 GPU → Save, then start again from Step 1. |
| Disconnected during Step 6 | Reconnect, run Steps 1–4, skip Step 5, run Step 6. Finished experiments are skipped. |
| `refusing to mix smoke-test and real results` | Step 5 must write to `/content/smoke_out`, never to the Drive folder. |
| `LEAKAGE` error | Stop. Re-run Step 4 from the start. |
| Out of memory | Runtime → Restart session, then Steps 1–4 and Step 6 again. |
| An experiment looks wrong | `python run_experiments.py --redo <ID> ...` runs just that one again. |